In [1]:
# Data Cleaning - Home Office Immigration Statistics
# extracting skilled worker visa grants per sector per year

import pandas as pd

# loading the occupation data sheet - same as EDA
df = pd.read_excel('HO_ImmigStats_13Jun2026.xlsx', sheet_name='Data_Occ_D01')

# fixing headers
df.columns = df.iloc[2]
df = df.drop([0, 1, 2]).reset_index(drop=True)

df.columns = ['Year', 'Quarter', 'Nationality', 'Region', 'Visa_Type',
              'Visa_Subgroup', 'Industry', 'Occ_Major', 'Occ_SubMajor',
              'Occ_Minor', 'Occ_Unit', 'Applications']

# filtering to Worker visa type only
df_worker = df[df['Visa_Type'] == 'Worker'].copy()
df_worker['Applications'] = pd.to_numeric(df_worker['Applications'], errors='coerce')

print("shape after filtering to Worker:", df_worker.shape)
print("\nunique years:", df_worker['Year'].unique())
print("\nunique industries:", df_worker['Industry'].unique())

shape after filtering to Worker: (48557, 12)

unique years: [2021 2022 2023 2024 2025 2026]

unique industries: ['Accommodation and Food Service Activities'
 'Administrative and Support Service Activities' 'Education'
 'Human Health and Social Work Activities'
 'Information and Communications'
 'Professional, Scientific and Technical Activities'
 'Public Admin and defence; compulsory social security'
 'Water supply; sewerage, waste management and remediation activities'
 'Wholesale and retail trade; repair of motor vehicles and motorcycles'
 'Activities of extraterritorial organisations and bodies'
 'Arts, Entertainment and Recreation' 'Construction'
 'Electricity, gas, steam and air conditioning supply'
 'Financial and Insurance Activities' 'Manufacturing'
 'Mining and Quarrying' 'Other Service Activities'
 'Real Estate Activities' 'Transportation and Storage'
 'Activities of households as employers; production activities of household for own use'
 'Agriculture, Forestry and Fishing'


In [2]:
# mapping industries to my 5 sectors
industry_to_sector = {
    'Information and Communications': 'Technology',
    'Human Health and Social Work Activities': 'Healthcare',
    'Financial and Insurance Activities': 'Finance',
    'Professional, Scientific and Technical Activities': 'Engineering',
    'Manufacturing': 'Engineering',
    'Education': 'Education'
}

# adding sector column
df_worker['Sector'] = df_worker['Industry'].map(industry_to_sector)

# keeping only my 5 sectors
df_sectors = df_worker[df_worker['Sector'].notna()].copy()

print("shape after sector filtering:", df_sectors.shape)
print("\nsector counts:")
print(df_sectors['Sector'].value_counts())

shape after sector filtering: (26900, 13)

sector counts:
Sector
Engineering    11672
Finance         4542
Healthcare      4139
Technology      4124
Education       2423
Name: count, dtype: int64


In [3]:
# aggregating to annual level per sector
df_annual = df_sectors.groupby(['Year', 'Sector'])['Applications'].sum().reset_index()

# renaming columns
df_annual.columns = ['Year', 'Sector', 'Visa_Grants']

# checking the result
print("annual visa grants per sector:")
print(df_annual.pivot(index='Year', columns='Sector', values='Visa_Grants'))

annual visa grants per sector:
Sector  Education  Engineering  Finance  Healthcare  Technology
Year                                                           
2021         4394        14562     7284       32358       14328
2022         5427        27439    12682       79869       22529
2023         6376        23924     9532      150735       15484
2024         4847        17409     8470       28232       12124
2025         2557        13727     8142       14064        9214
2026          390         2780     1812        1499        1934


In [4]:
# saving clean file
df_annual.to_csv('HO_ImmigStats_Clean.csv', index=False)
print("saved! shape:", df_annual.shape)
print("\nfirst 10 rows:")
print(df_annual.head(10))

saved! shape: (30, 3)

first 10 rows:
   Year       Sector  Visa_Grants
0  2021    Education         4394
1  2021  Engineering        14562
2  2021      Finance         7284
3  2021   Healthcare        32358
4  2021   Technology        14328
5  2022    Education         5427
6  2022  Engineering        27439
7  2022      Finance        12682
8  2022   Healthcare        79869
9  2022   Technology        22529


In [5]:
# notes for dissertation
print("key cleaning steps done:")
print("- filtered to Worker visa type only")
print("- mapped 6 industries to 5 sectors")
print("- aggregated to annual level per sector")
print("- 2026 is partial year (Q1 only) - flag in dissertation")
print("- Healthcare peaked at 150,735 in 2023 - key finding")
print("- saved as HO_ImmigStats_Clean.csv")

key cleaning steps done:
- filtered to Worker visa type only
- mapped 6 industries to 5 sectors
- aggregated to annual level per sector
- 2026 is partial year (Q1 only) - flag in dissertation
- Healthcare peaked at 150,735 in 2023 - key finding
- saved as HO_ImmigStats_Clean.csv
